In [ ]:
!pip install pyiwn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


import pyiwn
import pandas as pd
import os
import ipywidgets as widgets
from IPython.display import display


# ================== PROJECT SETUP ==================
PROJECT_DIR = "/content/drive/MyDrive/IndoWordNet_Project"
os.makedirs(PROJECT_DIR, exist_ok=True)

DATASET_FILE = os.path.join(PROJECT_DIR, "word.csv")

if os.path.exists(DATASET_FILE):
    df = pd.read_csv(DATASET_FILE)
else:
    df = pd.DataFrame(columns=["word","synset_id", "pos", "gloss", "examples", "synonyms"])


# ================== WORDNET ==================
iwn = pyiwn.IndoWordNet(lang=pyiwn.Language.KANNADA)


# ================== WIDGETS ==================
search_box = widgets.Text(description="ಕನ್ನಡ ಪದ:")
search_button = widgets.Button(description="Search", button_style="info")

word_field = widgets.Text(description="Word:", disabled=True)
synset_id_field = widgets.Text(description="Synset ID:", disabled=True)
pos_field = widgets.Text(description="POS:", disabled=True)

gloss_field = widgets.Textarea(description="Gloss:", disabled=True, layout=widgets.Layout(height="90px"))
example_field = widgets.Textarea(description="Examples:", disabled=True, layout=widgets.Layout(height="90px"))
synonym_field = widgets.Textarea(description="Synonyms:", disabled=True, layout=widgets.Layout(height="70px"))



edit_button = widgets.Button(description="Edit", button_style="warning")
save_button = widgets.Button(description="Save", button_style="success")
delete_button = widgets.Button(description="Delete", button_style="danger")

output = widgets.Output()


# ================== HELPERS ==================
def get_synset_info(word):
    synsets = iwn.synsets(word)
    if not synsets:
        return None

    syn = synsets[0]
    return {
        "word": word,
        "synset_id": syn.synset_id(),
        "pos": syn.pos(),
        "gloss": syn.gloss(),
        "examples": syn.examples(),
        "synonyms": syn.lemma_names()

    }


def clear_all_fields():
    word_field.value = ""
    pos_field.value = ""
    gloss_field.value = ""
    example_field.value = ""
    synonym_field.value = ""
    synset_id_field.value = ""


def enable_fields():
    word_field.disabled =False
    pos_field.disabled =False
    synset_id_field.disabled=False
    gloss_field.disabled = False
    example_field.disabled = False
    synonym_field.disabled = False


def disable_fields():
    word_field.disabled =True
    pos_field.disabled =True
    synset_id_field.disabled=True
    gloss_field.disabled = True
    example_field.disabled = True
    synonym_field.disabled = True


def load_from_dataset(word):
    row = df[df["word"] == word].iloc[0]
    word_field.value = row["word"]
    synset_id_field.value = str(row["synset_id"])
    pos_field.value = row["pos"]
    gloss_field.value = row["gloss"]
    example_field.value = row["examples"]
    synonym_field.value = row["synonyms"]
    disable_fields()


# ================== SEARCH ==================
def on_search(b):
    output.clear_output()
    clear_all_fields()

    word = search_box.value.strip()
    if not word:
        return

    # Normalize dataset
    df["word"] = df["word"].astype(str).str.strip()

    # ✅ Word exists → load for DELETE
    if word in df["word"].values:
        load_from_dataset(word)
        with output:
            print("ℹ️ Word already exists.")
        return

    # 🔍 WordNet lookup
    data = get_synset_info(word)
    if not data:
        with output:
            print(f"❌ No data found for '{word}'")
        return

    word_field.value = data["word"]
    synset_id_field.value = str(data["synset_id"])
    pos_field.value = data["pos"]
    gloss_field.value = data["gloss"]
    example_field.value = ", ".join(data["examples"])
    synonym_field.value = ", ".join(data["synonyms"])


    disable_fields()
    with output:
        print("✅ WordNet data loaded")


# ================== EDIT ==================
def edit_data(b):
    if not word_field.value:
        return
    enable_fields()
    output.clear_output()
    with output:
        print("✏️ Edit mode enabled")


# ================== SAVE ==================
def save_data(b):
    global df

    word = word_field.value.strip()
    if not word:
        return

    if word in df["word"].values:
        output.clear_output()
        with output:
            print("❌ Duplicate word not supported")
        return

    new_row = {
        "word": word,
        "synset_id": synset_id_field.value,
        "pos": pos_field.value,
        "gloss": gloss_field.value,
        "examples": example_field.value,
        "synonyms": synonym_field.value

    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df = df.sort_values("word")
    df.to_csv(DATASET_FILE, index=False)

    disable_fields()
    clear_all_fields()
    output.clear_output()
    with output:
        print("💾 Data saved successfully")


# ================== DELETE ==================
def delete_data(b):
    global df

    word = word_field.value.strip()
    if not word:
        return

    df["word"] = df["word"].astype(str).str.strip()

    if word not in df["word"].values:
        output.clear_output()
        with output:
            print("❌ Word not found in dataset")
        return

    df = df[df["word"] != word]
    df.to_csv(DATASET_FILE, index=False)

    clear_all_fields()
    output.clear_output()
    with output:
        print("🗑️ Word deleted successfully")



# ================== EVENTS ==================
search_button.on_click(on_search)
edit_button.on_click(edit_data)
save_button.on_click(save_data)
delete_button.on_click(delete_data)


# ================== UI ==================
display(
    widgets.VBox([
        widgets.HBox([search_box, search_button]),
        word_field,
        synset_id_field,
        pos_field,
        gloss_field,
        example_field,
        synonym_field,
        widgets.HBox([edit_button, save_button, delete_button]),
        output
    ])
)

In [ ]:
import pyiwn

# Load Kannada WordNet
iwn = pyiwn.IndoWordNet(lang=pyiwn.Language.KANNADA)

filtered_words = set()   # avoid duplicates

for synset in iwn.all_synsets():
    for lemma in synset.lemmas():
        word = lemma.name()
        if '_' not in word and '-' not in word:
            filtered_words.add(word)

print("Total unique Kannada words without _ and - :", len(filtered_words))

for word in filtered_words:
    print(word)


Total unique Kannada words without _ and - : 39888
ಮುವತ್ತೊಂದನೇ
ಬೇರೆದೇಶದ
ಗ್ರಹಿಸಬಲ್ಲ
ರಾಮರಾಜ್ಯ
ಕೇಳಿದಷ್ಟು
ಸಂಗ್ರಹಕರ್ತ
ಕಂಕಾಲ
ಪರಿವ್ಯಾಪಕ
ಸಂವೇದನಾಶೀಲವಾದಂತಹ
ಅಂಜಲಿ
ಕಮರಿಕೆ
ಸಮ್ಮೋಹನಗೊಳಿಸಬಲ್ಲಂತ
ಹಠಯೋಗಿಯಾದ
ಬಲಹೀನ
ಸನ್ಮಾನಿಸದ
ಆಕಾರಿತವಾದಂತ
ತರಲ್ಪಟ್ಟಂತ
ಸಮನಲ್ಲದಂತ
ಮುಂದೆ
ಉಯ್ಯಾಲೆಯಾಡು
ಪಡಸಾಲೆ
ಆಯ್ಕೆಯಾದಂತ
ಕುತೂಹಲಕಾರಿಯಾದ
ಯಾಚಿಸಲಾದಂತಹ
ಒಂದು
ಕೈದಾದಂತ
ಪಾರತಂತ್ರ್ಯ
ರಕ್ತಕುಡಿಯುವ
ಹೆಸರುವೆತ್ತ
ಬದಲಾಗು
ಹತ್ತಿದ
ಅಲ್ಲಾಡುವ
ಕ್ಷಮಾಹೀನ
ಕೌಮಾರಿ
ಜನ್ಮಕೊಡು
ತೊಡಕಿರದ
ಬಿಗಿಯಾದಂತಹ
ವ್ಯವಸ್ಥೆಯಂತಹ
ಮುಜು
ಟೆಕ್ನಾಲಜಿ
ಸ್ವಶಿಕ್ಷಿತವಾದಂತಹ
ಪಂಕ್ತಿ
ತೇಗು
ಸೋತುಹೋದಂತ
ಚೆಲ್ಲಿರುವ
ಕಾತುರತೆ
ಭೃಂಗ
ಕ್ಯಾರಟ್
ಶೀತಲವಾಗು
ವಿಕಿರಣಪಟು
ರಾಜದ್ರೋಹಿಯಾದಂತ
ಸುಂದರಿಯಾದಂತಹ
ಹಚ್ಚಹಸಿರಾದ
ಮಾಖವಾಡ
ಲಿಬೆರಿಯನ್
ಸಪ್ತಸ್ವರ
ಅಲ್ಪವ್ಯವಯಿ
ಸರಿಸಾಟಿಯಿಲ್ಲದ
ಪರಿಪೂರ್ಣವಾದಂತ
ಕಾಲ್ಗವಸು
ಆಮಂತ್ರಿಕ
ಮೊದಲಿನಂತೆ
ಆಟವಾಡಬೇಕು
ಅಕರಣ
ಶೌಚಗೃಹ
ಅವರಪ್ರಭು
ಮಾನ್ಸೂನಿನಂತಹ
ಮುವತ್ತನಾಲ್ಕು
ಮಾಧ್ಯಂದಿನ
ಟಿಕ್ಕಿಸುವವ
ಅರಣ್ಯವಾಸಿಯಾದಂತ
ಹೆಸರಿನಂತ
ಅಭಾಜ್ಯವಾದಂತ
ಅವತಾರವೆತ್ತಂತ
ಅಜರಾಮರ
ಜೀವಮಾನ
ಪಥ್ಯಾಹಾರ
ಮೆಲುಧ್ವನಿಯ
ವೇಷ್ಯೆ
ಸತ್ತಸಂಧನಾದಂತ
ತುಲಾಧಾನ
ಶಾಂತಿದಾಯಕವಾದಂತಹ
ಶನಿವಾರ
ಹೇಳು
ಶುಭಚಿಂತಕ
ಉಪಕಾರಗೇಡಿಯಾದಂತಹ
ಯೋಗಿಯಲ್ಲದಂತ
ಗಿರಿಧಾರಿ
ಅಭಿವಂದನೀಯವಾದಂತಹ
ಸ್ಮಶಾಣ
ಇಂದ್ರಿಯಾ
ಪತ್ತೇದಾರಿಕೆಯಂತಹ
ಸಂಹಾರಕವಾದ
ಅವರೋಹಿತ
ಹೊಂದಿಸಬಲ್